# CNN Training - Exercise Binary Classification

Classification binaire (correct / incorrect) des réponses d'exercices d'examen.

**Modèle** : EfficientNetV2-S pré-entraîné sur ImageNet, avec fine-tuning de la tête de classification.

**Validation** : Stratified 5-Fold Cross Validation

**Gestion du déséquilibre** : class weights calculés automatiquement

## 1. Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 2. Configuration

In [ ]:
# ===== À MODIFIER =====
DATASET_ROOT = "../data/labeled/by_exercise"   # dossier avec q01/, q02/, ... contenant correct/ et incorrect/
OUTPUT_DIR = "../outputs/exercise_cnn"    # dossier de sortie (modèles, résultats)
# ======================

# Hyperparamètres
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 1e-4
N_FOLDS = 5
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

## 3. Chargement du dataset

On parcourt la structure `by_exercise/q01/correct/`, `by_exercise/q01/incorrect/`, etc. pour collecter tous les chemins d'images et leurs labels.

In [ ]:
X = np.load("../data/processed/X_exercise.npy")
y = np.load("../data/processed/y_exercise.npy")

print(f"Dataset shape : {X.shape}")
print(f"Correct : {np.sum(y == 1)}")
print(f"Incorrect : {np.sum(y == 0)}")

## 4. Dataset PyTorch et augmentation

**Augmentation (entraînement)** : rotations légères, flip horizontal, variation de luminosité/contraste. On reste subtil car ce sont des scans, pas des photos naturelles.

**Validation** : pas d'augmentation, juste resize + normalisation ImageNet.

In [ ]:
class ExamDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).permute(0, 3, 1, 2)  # (N,H,W,C) → (N,C,H,W)
        self.y = torch.tensor(y, dtype=torch.float32)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## 5. Modèle

EfficientNetV2-S (2021) pré-entraîné sur ImageNet. On remplace la dernière couche fully-connected par une tête de classification binaire.

**Stratégie de fine-tuning** :
1. On gèle tout le backbone (features pré-entraînées)
2. On entraîne uniquement la tête de classification
3. Après quelques epochs, on dégèle les dernières couches du backbone pour affiner

In [ ]:
def create_model(unfreeze_last_n=0):
    """Crée un EfficientNetV2-S avec tête de classification binaire."""
    model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
    
    # Geler le backbone
    for param in model.features.parameters():
        param.requires_grad = False
    
    # Dégeler les N derniers blocs si demandé
    if unfreeze_last_n > 0:
        for param in model.features[-unfreeze_last_n:].parameters():
            param.requires_grad = True
    
    # Nouvelle tête de classification
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, 128),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(128, 1)
    )
    
    return model.to(device)

# Test
model_test = create_model()
trainable = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_test.parameters())
print(f"Paramètres entraînables : {trainable:,} / {total:,} ({trainable/total*100:.1f}%)")
del model_test

## 6. Fonctions d'entraînement et d'évaluation

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(imgs).squeeze(1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * imgs.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs).squeeze(1)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * imgs.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc = (all_preds == all_labels).mean()
    
    return total_loss / len(all_labels), acc, all_preds, all_labels

## 7. Entraînement avec K-Fold Cross Validation

Pour chaque fold :
1. Phase 1 (epochs 1-15) : backbone gelé, on entraîne que la tête
2. Phase 2 (epochs 16-30) : on dégèle les 2 derniers blocs du backbone pour affiner

Les class weights compensent le déséquilibre correct/incorrect.

In [ ]:
# Class weights pour le déséquilibre
n_correct = np.sum(all_labels == 1)
n_incorrect = np.sum(all_labels == 0)
weight_for_incorrect = len(all_labels) / (2 * n_incorrect)
weight_for_correct = len(all_labels) / (2 * n_correct)

print(f"Class weights : correct={weight_for_correct:.2f}, incorrect={weight_for_incorrect:.2f}")
print(f"(la classe minoritaire a un poids plus élevé)")

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_results = []
all_fold_preds = np.zeros(len(all_labels))
histories = []

UNFREEZE_AT_EPOCH = 15  # dégeler les dernières couches après cette epoch

for fold, (train_idx, val_idx) in enumerate(skf.split(, all_labels)):
    print(f"\n{'='*50}")
    print(f"FOLD {fold + 1}/{N_FOLDS}")
    print(f"Train: {len(train_idx)} | Val: {len(val_idx)}")
    
    # Datasets
    train_labels = all_labels[train_idx]
    val_labels = all_labels[val_idx]
    
    train_ds = ExamDataset(X[train_idx], y[train_idx])
    val_ds = ExamDataset(X[val_idx], y[val_idx])
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    # Modèle (backbone gelé au départ)
    model = create_model(unfreeze_last_n=0)
    
    # Loss avec class weights
    pos_weight = torch.tensor([weight_for_incorrect / weight_for_correct]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    
    best_val_acc = 0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    
    for epoch in range(EPOCHS):
        # Phase 2 : dégeler les dernières couches
        if epoch == UNFREEZE_AT_EPOCH:
            print(f"  → Dégel des 2 derniers blocs du backbone")
            for param in model.features[-2:].parameters():
                param.requires_grad = True
            optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE * 0.1)
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
        
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step(val_loss)
        
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"{OUTPUT_DIR}/best_model_fold{fold+1}.pth")
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:2d}/{EPOCHS} | "
                  f"Train loss: {train_loss:.4f} acc: {train_acc:.3f} | "
                  f"Val loss: {val_loss:.4f} acc: {val_acc:.3f}")
    
    # Évaluation finale du fold avec le meilleur modèle
    model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best_model_fold{fold+1}.pth"))
    _, final_acc, preds, labels = evaluate(model, val_loader, criterion)
    
    all_fold_preds[val_idx] = preds
    fold_results.append(final_acc)
    histories.append(history)
    
    print(f"  Best val accuracy: {best_val_acc:.3f}")

print(f"\n{'='*50}")
print(f"RÉSULTATS K-FOLD")
print(f"Accuracy par fold : {[f'{a:.3f}' for a in fold_results]}")
print(f"Moyenne : {np.mean(fold_results):.3f} ± {np.std(fold_results):.3f}")

## 8. Courbes d'entraînement

Loss et accuracy pour chaque fold. On vérifie qu'il n'y a pas d'overfitting (train qui monte, val qui stagne ou descend).

In [ ]:
fig, axes = plt.subplots(N_FOLDS, 2, figsize=(14, 4 * N_FOLDS))

for i, history in enumerate(histories):
    axes[i, 0].plot(history["train_loss"], label="Train")
    axes[i, 0].plot(history["val_loss"], label="Val")
    axes[i, 0].set_title(f"Fold {i+1} - Loss")
    axes[i, 0].legend()
    axes[i, 0].set_xlabel("Epoch")
    axes[i, 0].axvline(x=UNFREEZE_AT_EPOCH, color='gray', linestyle='--', alpha=0.5, label='Unfreeze')
    
    axes[i, 1].plot(history["train_acc"], label="Train")
    axes[i, 1].plot(history["val_acc"], label="Val")
    axes[i, 1].set_title(f"Fold {i+1} - Accuracy")
    axes[i, 1].legend()
    axes[i, 1].set_xlabel("Epoch")
    axes[i, 1].axvline(x=UNFREEZE_AT_EPOCH, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_curves.png", dpi=150, bbox_inches='tight')
plt.show()

## 9. Évaluation globale

Matrice de confusion et classification report sur l'ensemble du dataset (chaque image a été prédite exactement une fois via le K-Fold).

In [ ]:
print("Classification Report (agrégé sur les 5 folds) :\n")
print(classification_report(all_labels, all_fold_preds, 
                            target_names=["Incorrect", "Correct"]))

cm = confusion_matrix(all_labels, all_fold_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["Incorrect", "Correct"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Blues')
plt.title("Matrice de confusion (K-Fold agrégé)")
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

## 10. Entraînement du modèle final

On ré-entraîne un modèle sur **toutes** les données (pas de split validation) pour avoir le modèle le plus performant possible pour le déploiement. Le K-Fold a donné l'estimation de performance, ce modèle-ci est celui qu'on garde.

In [ ]:
print("Entraînement du modèle final sur toutes les données...")

full_ds = ExamDataset(X, y)
full_loader = DataLoader(full_ds, batch_size=BATCH_SIZE, shuffle=True)

final_model = create_model(unfreeze_last_n=0)
pos_weight = torch.tensor([weight_for_incorrect / weight_for_correct]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, final_model.parameters()), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    if epoch == UNFREEZE_AT_EPOCH:
        print(f"  → Dégel des 2 derniers blocs")
        for param in final_model.features[-2:].parameters():
            param.requires_grad = True
        optimizer = optim.Adam(filter(lambda p: p.requires_grad, final_model.parameters()), lr=LEARNING_RATE * 0.1)
    
    train_loss, train_acc = train_one_epoch(final_model, full_loader, criterion, optimizer)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:2d}/{EPOCHS} | Loss: {train_loss:.4f} | Acc: {train_acc:.3f}")

torch.save(final_model.state_dict(), f"{OUTPUT_DIR}/final_model.pth")
print(f"\nModèle final sauvegardé dans {OUTPUT_DIR}/final_model.pth")

## 11. Récapitulatif

In [ ]:
print("RÉCAPITULATIF")
print(f"  Modèle : EfficientNetV2-S (transfer learning ImageNet)")
print(f"  Dataset : {len(all_labels)} images ({np.sum(all_labels==1)} correct, {np.sum(all_labels==0)} incorrect)")
print(f"  Validation : {N_FOLDS}-Fold Stratified Cross Validation")
print(f"  Accuracy moyenne : {np.mean(fold_results):.3f} ± {np.std(fold_results):.3f}")
print(f"  Epochs : {EPOCHS} (dégel à epoch {UNFREEZE_AT_EPOCH})")
print(f"\nFichiers générés :")
for f in sorted(Path(OUTPUT_DIR).glob("*")):
    size = f.stat().st_size / 1e6
    print(f"  {f.name} ({size:.1f} MB)")